# 5. 사양표는 믿을 만한가 — 리뷰 키워드로 본 사양표 신뢰도

**찾을 현상**
- 사양표가 있어도 성능 문제(최적화·프레임·끊김) 불만이 유독 많은 게임이 있다
- 출시 후 몇 개월 지나면 성능 문제 불만이 줄어든다

**인사이트**: 게임 상세의 사양표 옆에 '이 사양표를 믿어도 되는지'를 표시 (B: spec_trust.json). 초기엔 문제였지만 지금은 나아진 게임도 가려냄

**데이터**: Steam 한국어 리뷰 본문(ES 키워드), 리뷰 작성 시각, Steam Deck 여부. 한계: 리뷰어가 권장 사양을 충족했는지는 모른다

> 지금은 `mock` 데이터(가설을 심은 가짜 값)라 결과는 결론이 아닙니다. 정제 테이블이 채워지면 `load_dataset("db")`로 바꾸세요.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from gamenuri_analysis import a05_spec_trust as m
from gamenuri_analysis.loader import load_dataset
from gamenuri_analysis.report import fig_dir

In [ ]:
data = load_dataset("mock")  # 실데이터: load_dataset("db") (MariaDB Replica, 읽기 전용)
data.source

## 계산 · 그래프
`run()`이 계산하고 `analysis/out/figures/<분석>/`에 PNG와 summary.json을 저장합니다.

In [ ]:
res = m.run(data)
for t in res.takeaways:
    print('-', t)

In [ ]:
for f in res.figures:
    print(f['title'], '—', f['caption'])
    display(Image(filename=str(fig_dir(m.KEY) / f['file'])))

In [ ]:
for t in res.tables:
    print(t['title'])
    display(pd.DataFrame(t['rows'], columns=t['columns']))

## 직접 들여다보기
`res.dfs`에 중간 결과가 들어 있어요. 아래 셀을 고쳐 가며 탐색해 보세요.

In [ ]:
print(list(res.dfs))
res.dfs['trust'].sort_values('recent_share', ascending=False).head(10)[['name','recent_share','launch_share','top_pct','verdict']]

## 서비스로 넘기는 것
- 게임별 태그(`res.per_game`) → B: `out/api/game_insights.json` → 게임 상세 화면
- 구매 타이밍 규칙(`res.api`) → B: `out/api/timing_rules.json` → Backend


In [ ]:
print(sum(len(v) for v in res.per_game.values()), '개 태그')
res.api